In [ ]:
# cell 1
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedShuffleSplit, ParameterGrid
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score,
    f1_score
)
from imblearn.over_sampling import RandomOverSampler


In [ ]:
# cell 2
# Settings

DATA_DIR = r"..\Data\model_data"
RESULTS_DIR = r"..\Results\Model_selection"


SEED = 10
N_SPLITS = 5

OUTCOME_PREFIXES = [
    "mask",
    "protective",
    "social_avoidance"
]

cv = StratifiedShuffleSplit(
    n_splits=N_SPLITS,
    test_size=1 / N_SPLITS,
    random_state=SEED
)



In [ ]:
# cell 3
# Parameter grid

tree_param_grid = {
    "max_depth": [3, 5, 7, 10, 15],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "min_weight_fraction_leaf": [0.0, 0.01, 0.02],
    "criterion": ["gini", "entropy"],
    "splitter": ["best"],
    "min_impurity_decrease": [0.0, 0.0001, 0.001, 0.01],
    "class_weight": [None]
}



In [ ]:
# cell 4
# Helper functions

def load_training_data(prefix):
    X_train = pd.read_csv(f"{DATA_DIR}\\{prefix}_X_train.csv")
    y_train = pd.read_csv(f"{DATA_DIR}\\{prefix}_y_train.csv").values.ravel()

    return X_train, y_train


def prepare_fold_data(X_train, X_val):
    X_train = X_train.copy()
    X_val = X_val.copy()

    bool_cols_train = X_train.select_dtypes(include=["bool"]).columns
    bool_cols_val = X_val.select_dtypes(include=["bool"]).columns

    if len(bool_cols_train) > 0:
        X_train[bool_cols_train] = X_train[bool_cols_train].astype(int)

    if len(bool_cols_val) > 0:
        X_val[bool_cols_val] = X_val[bool_cols_val].astype(int)

    X_train, X_val = X_train.align(
        X_val,
        join="left",
        axis=1,
        fill_value=0
    )

    return X_train, X_val


def summarize_cv_scores(cv_scores):
    summary_df = pd.DataFrame({
        "fold": cv_scores["fold"],
        "precision": cv_scores["precision"],
        "recall": cv_scores["recall"],
        "roc_auc": cv_scores["roc_auc"],
        "accuracy": cv_scores["accuracy"],
        "f1": cv_scores["f1"]
    })

    mean_row = pd.DataFrame([{
        "fold": "mean",
        "precision": summary_df["precision"].mean(),
        "recall": summary_df["recall"].mean(),
        "roc_auc": summary_df["roc_auc"].mean(),
        "accuracy": summary_df["accuracy"].mean(),
        "f1": summary_df["f1"].mean()
    }])

    se_row = pd.DataFrame([{
        "fold": "se",
        "precision": summary_df["precision"].std(ddof=1) / np.sqrt(N_SPLITS),
        "recall": summary_df["recall"].std(ddof=1) / np.sqrt(N_SPLITS),
        "roc_auc": summary_df["roc_auc"].std(ddof=1) / np.sqrt(N_SPLITS),
        "accuracy": summary_df["accuracy"].std(ddof=1) / np.sqrt(N_SPLITS),
        "f1": summary_df["f1"].std(ddof=1) / np.sqrt(N_SPLITS)
    }])

    summary_df = pd.concat(
        [summary_df, mean_row, se_row],
        ignore_index=True
    )

    return summary_df


def get_mean_se(summary_df, metric):
    mean_value = summary_df.loc[
        summary_df["fold"] == "mean",
        metric
    ].values[0]

    se_value = summary_df.loc[
        summary_df["fold"] == "se",
        metric
    ].values[0]

    return mean_value, se_value


def print_mean_se(summary_df, prefix):
    print("\n" + "=" * 60)
    print(f"Classification Tree results for: {prefix}")

    for metric in ["precision", "recall", "roc_auc", "accuracy", "f1"]:
        mean_value, se_value = get_mean_se(summary_df, metric)
        print(f"{metric}: {mean_value:.3f} ± {se_value:.3f}")



In [ ]:
# cell 5
# Model evaluation

def evaluate_tree_params(prefix, params, upsample=True):
    X, y = load_training_data(prefix)

    cv_scores = {
        "fold": [],
        "precision": [],
        "recall": [],
        "roc_auc": [],
        "accuracy": [],
        "f1": []
    }

    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):

        X_train_fold = X.iloc[train_idx].copy()
        y_train_fold = y[train_idx].copy()

        X_val_fold = X.iloc[val_idx].copy()
        y_val_fold = y[val_idx].copy()

        if upsample:
            upsampler = RandomOverSampler(random_state=SEED)
            X_train_fold, y_train_fold = upsampler.fit_resample(
                X_train_fold,
                y_train_fold
            )

            if not isinstance(X_train_fold, pd.DataFrame):
                X_train_fold = pd.DataFrame(
                    X_train_fold,
                    columns=X.columns
                )

        X_train_prepared, X_val_prepared = prepare_fold_data(
            X_train_fold,
            X_val_fold
        )

        model = DecisionTreeClassifier(
            random_state=SEED,
            **params
        )

        model.fit(X_train_prepared, y_train_fold)

        y_pred = model.predict(X_val_prepared)
        y_prob = model.predict_proba(X_val_prepared)[:, 1]

        cv_scores["fold"].append(fold_idx)
        cv_scores["precision"].append(
            precision_score(y_val_fold, y_pred, zero_division=0)
        )
        cv_scores["recall"].append(
            recall_score(y_val_fold, y_pred, zero_division=0)
        )
        cv_scores["roc_auc"].append(
            roc_auc_score(y_val_fold, y_prob)
        )
        cv_scores["accuracy"].append(
            accuracy_score(y_val_fold, y_pred)
        )
        cv_scores["f1"].append(
            f1_score(y_val_fold, y_pred, zero_division=0)
        )

    summary = {
        "params": params,
        "precision_mean": np.mean(cv_scores["precision"]),
        "recall_mean": np.mean(cv_scores["recall"]),
        "roc_auc_mean": np.mean(cv_scores["roc_auc"]),
        "accuracy_mean": np.mean(cv_scores["accuracy"]),
        "f1_mean": np.mean(cv_scores["f1"])
    }

    return cv_scores, summary



In [ ]:
# cell 6
# Hyperparameter tuning

def tune_classification_tree(prefix, upsample=True):
    all_results = []

    total_combinations = len(list(ParameterGrid(tree_param_grid)))
    print("\n" + "=" * 60)
    print(f"Tuning Classification Tree for: {prefix}")
    print(f"Total parameter combinations: {total_combinations}")
    print(f"Upsampling: {upsample}")

    for i, params in enumerate(ParameterGrid(tree_param_grid), start=1):
        _, summary = evaluate_tree_params(
            prefix=prefix,
            params=params,
            upsample=upsample
        )

        all_results.append(summary)

        if i % 100 == 0:
            print(f"Finished {i} / {total_combinations} combinations")

    results_df = pd.DataFrame(all_results)

    results_df = results_df.sort_values(
        by=["roc_auc_mean", "f1_mean", "accuracy_mean"],
        ascending=False
    ).reset_index(drop=True)

    best_row = results_df.iloc[0]
    best_params = best_row["params"]

    print(f"\nBest params for {prefix}:")
    print(best_params)
    print("Best mean ROC AUC:", round(best_row["roc_auc_mean"], 3))
    print("Best mean precision:", round(best_row["precision_mean"], 3))
    print("Best mean recall:", round(best_row["recall_mean"], 3))
    print("Best mean accuracy:", round(best_row["accuracy_mean"], 3))
    print("Best mean F1:", round(best_row["f1_mean"], 3))

    results_df.to_csv(
        f"{RESULTS_DIR}\\tree_{prefix}_tuning_results.csv",
        index=False
    )

    return results_df, best_params



In [ ]:
# cell 7
# Cross-validate best tree

def cross_validate_best_tree(prefix, best_params, upsample=True):
    cv_scores, _ = evaluate_tree_params(
        prefix=prefix,
        params=best_params,
        upsample=upsample
    )

    summary_df = summarize_cv_scores(cv_scores)

    summary_df.to_csv(
        f"{RESULTS_DIR}\\tree_{prefix}_cv_summary.csv",
        index=False
    )

    print_mean_se(summary_df, prefix)

    return summary_df



In [ ]:
# cell 8
# Run tree tuning and CV

tree_results = {}
tree_best_params = {}
tree_summaries = {}

for prefix in OUTCOME_PREFIXES:
    results_df, best_params = tune_classification_tree(
        prefix=prefix,
        upsample=True
    )

    summary_df = cross_validate_best_tree(
        prefix=prefix,
        best_params=best_params,
        upsample=True
    )

    tree_results[prefix] = results_df
    tree_best_params[prefix] = best_params
    tree_summaries[prefix] = summary_df




Tuning Classification Tree for: mask
Total parameter combinations: 1920
Upsampling: True
Finished 100 / 1920 combinations
Finished 200 / 1920 combinations
Finished 300 / 1920 combinations
Finished 400 / 1920 combinations
Finished 500 / 1920 combinations
Finished 600 / 1920 combinations
Finished 700 / 1920 combinations
Finished 800 / 1920 combinations
Finished 900 / 1920 combinations
Finished 1000 / 1920 combinations
Finished 1100 / 1920 combinations
Finished 1200 / 1920 combinations
Finished 1300 / 1920 combinations
Finished 1400 / 1920 combinations
Finished 1500 / 1920 combinations
Finished 1600 / 1920 combinations
Finished 1700 / 1920 combinations
Finished 1800 / 1920 combinations
Finished 1900 / 1920 combinations

Best params for mask:
{'class_weight': None, 'criterion': 'gini', 'max_depth': 10, 'min_impurity_decrease': 0.0001, 'min_samples_leaf': 10, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'splitter': 'best'}
Best mean ROC AUC: 0.87
Best mean precision: 0.834
Best

In [ ]:
# cell 9
# Build comparison table

rows = []

for prefix, summary_df in tree_summaries.items():
    row = {
        "model": "Classification Tree",
        "outcome": prefix,
        "best_params": tree_best_params[prefix]
    }

    for metric in ["precision", "recall", "roc_auc", "accuracy", "f1"]:
        mean_value, se_value = get_mean_se(summary_df, metric)

        row[metric] = mean_value
        row[f"{metric}_se"] = se_value
        row[f"{metric}_mean_se"] = f"{mean_value:.3f} ± {se_value:.3f}"

    rows.append(row)

tree_comparison_df = pd.DataFrame(rows)

tree_comparison_df.to_csv(
    f"{RESULTS_DIR}\\tree_cv_comparison.csv",
    index=False
)

tree_comparison_df

,model,outcome,best_params,precision,precision_se,precision_mean_se,recall,recall_se,recall_mean_se,roc_auc,roc_auc_se,roc_auc_mean_se,accuracy,accuracy_se,accuracy_mean_se,f1,f1_se,f1_mean_se
0,Classification Tree,mask,"{'class_weight': None, 'criterion': 'gini', 'm...",0.834346,0.003855,0.834 ± 0.004,0.794343,0.008408,0.794 ± 0.008,0.870427,0.001485,0.870 ± 0.001,0.803831,0.002633,0.804 ± 0.003,0.813689,0.003404,0.814 ± 0.003
1,Classification Tree,protective,"{'class_weight': None, 'criterion': 'gini', 'm...",0.841589,0.004084,0.842 ± 0.004,0.688767,0.016877,0.689 ± 0.017,0.798610,0.002339,0.799 ± 0.002,0.712488,0.006847,0.712 ± 0.007,0.756909,0.009062,0.757 ± 0.009
2,Classification Tree,social_avoidance,"{'class_weight': None, 'criterion': 'entropy',...",0.782735,0.002268,0.783 ± 0.002,0.673101,0.006342,0.673 ± 0.006,0.771883,0.003242,0.772 ± 0.003,0.696605,0.002158,0.697 ± 0.002,0.723697,0.003165,0.724 ± 0.003
